# Multi-Size NMT Final Models (10M, 30M, 50M, 60M)

This notebook redefines a readable Transformer NMT architecture, validates parameter targets, trains across model sizes with size-specific batch policies, and saves final checkpoints as `final_{x}m_model.pt`.

In [18]:
import gc
import math
import os
import random
import time
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Sampler

from contextlib import nullcontext

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Removed safety check so CUDA executes strictly as requested
# User recently ran nvidia-smi, signifying GPU issues may be resolved

print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    if hasattr(torch.backends, "cuda") and hasattr(torch.backends.cuda, "matmul"):
        torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {props.total_memory / 1e9:.1f} GB")
    print(
        f"TF32 matmul={getattr(torch.backends.cuda.matmul, 'allow_tf32', False)} | "
        f"cuDNN benchmark={torch.backends.cudnn.benchmark}"
    )

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Ti | VRAM: 16.6 GB
TF32 matmul=True | cuDNN benchmark=True


In [19]:
# Define where data, cache, and trained models (checkpoints) will be stored
DATA_DIR = Path("/home/user35/nmt_bundle")
CACHE_DIR = DATA_DIR / "cache"
CHECKPOINT_DIR = CACHE_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)  # create folder if not exists

# Tokenization Settings
# Vocabulary size and maximum sequence length used by the model
VOCAB_SIZE = 32000
MAX_SEQ_LEN = 128

# Special tokens used in sequence modeling
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3

# --- Tokenizer Training (Patch) ---
# If the tokenizer doesn't exist, we can train it from the parquet files.
import pyarrow.parquet as pq

MODEL_PREFIX = str(CACHE_DIR / "nmt_unigram_v1_32000")
EXPECTED_MODEL = CACHE_DIR / "nmt_unigram_v1_32000.model"

if not EXPECTED_MODEL.exists():
    print("Tokenizer not found. Training a new Unigram model from parquet files...")
    
    # 1. Export sample text to a temporary text file for SentencePiece
    temp_txt_path = CACHE_DIR / "spm_train.txt"
    parquet_files = sorted(DATA_DIR.glob("train-*.parquet"))
    
    # Extract lines to train the tokenizer 
    max_lines_for_spm = 1_000_000
    lines_written = 0
    
    with open(temp_txt_path, "w", encoding="utf-8") as f:
        for p_file in parquet_files:
            if lines_written >= max_lines_for_spm:
                break   
            table = pq.read_table(str(p_file), columns=["src", "tgt"])
            for src, tgt in zip(table["src"].to_pylist(), table["tgt"].to_pylist()):
                if src and tgt:
                    f.write(src.strip() + "\n")
                    f.write(tgt.strip() + "\n")
                    lines_written += 2
                    if lines_written >= max_lines_for_spm:
                        break
                        
    # 2. Train the SentencePiece model
    print(f"Extracted {lines_written} lines. Training SentencePiece...")
    spm.SentencePieceTrainer.train(
        input=str(temp_txt_path),
        model_prefix=MODEL_PREFIX,
        vocab_size=VOCAB_SIZE,
        model_type='unigram',
        character_coverage=0.9995,
        pad_id=PAD_ID, unk_id=UNK_ID, bos_id=BOS_ID, eos_id=EOS_ID,
        pad_piece="<pad>", unk_piece="<unk>", bos_piece="<s>", eos_piece="</s>"
    )
    print(f"Tokenizer trained successfully: {EXPECTED_MODEL}")
    
    # 3. Clean up the temp file
    if temp_txt_path.exists():
        temp_txt_path.unlink()


# Tokenizer Loading
# Try loading a pre-trained SentencePiece tokenizer (unigram or BPE)
TOKENIZER_CANDIDATES = [
    CACHE_DIR / "nmt_unigram_v1_32000.model",
    CACHE_DIR / "nmt_bpe.model",
]

# Pick the first available tokenizer file
SP_MODEL_PATH = next((p for p in TOKENIZER_CANDIDATES if p.exists()), None)

# Fail early if tokenizer is missing (important for reproducibility)
if SP_MODEL_PATH is None:
    raise FileNotFoundError("No tokenizer model found in cache/")

# Load tokenizer → converts text ↔ tokens
sp = spm.SentencePieceProcessor(model_file=str(SP_MODEL_PATH))


# ---------- Model Size Targets ----------
# We train multiple models to study scaling effects
TARGET_MILLIONS = [10, 30]


# ---------- Batch Policy (Memory Management) ----------
# Each model size uses different batch settings to fit GPU memory
BATCH_POLICY = {
    "10M": {"max_tokens": 9000, "grad_accum": 2},
    "30M": {"max_tokens": 6200, "grad_accum": 4},
    "50M": {"max_tokens": 4400, "grad_accum": 6},
    "60M": {"max_tokens": 3200, "grad_accum": 8},
}
# Key idea: larger models → smaller batches → more gradient accumulation


# Training Configuration
# Global hyperparameters controlling training behavior
TRAINING_DEFAULTS = {
    "epochs_per_model": 20,                # how many times we train over dataset
    "max_train_batches_per_epoch": 7000,  # limit batches (for faster experiments)
    "max_val_batches": 20,                # limit validation batches

    "lr": 5e-4,                           # learning rate
    "warmup_steps": 3000,                 # gradual LR increase at start
    "weight_decay": 0.01,                 # regularization to prevent overfitting
    "label_smoothing": 0.1,               # prevents overconfidence
    "grad_clip": 1.0,                     # avoids exploding gradients

    "use_amp": DEVICE.type == "cuda",     # mixed precision for faster GPU training

    "target_tokens_per_sec": 100_000,     # performance target (speed)
    "oom_retry_limit": 6,                 # max retries if GPU runs out of memory

    "min_max_tokens": 1024,               # minimum batch size limit
    "auto_batch_shrink_factor": 0.85,     # reduce batch size if OOM
    "max_grad_accum": 20,                 # limit gradient accumulation

    "num_workers": min(8, os.cpu_count() or 1),  # parallel data loading
    "prefetch_factor": 2,                 # improves data loading speed
}


# Logging (Important for Explanation)
print(f"Tokenizer: {SP_MODEL_PATH.name} | vocab={sp.get_piece_size()}")

print("Targets:", TARGET_MILLIONS)

print("Batch policy:", BATCH_POLICY)

print(
    f"Perf target: {TRAINING_DEFAULTS['target_tokens_per_sec']:,} tok/s | "
    f"OOM retries/model: {TRAINING_DEFAULTS['oom_retry_limit']} | "
    f"workers={TRAINING_DEFAULTS['num_workers']}"
)


Tokenizer: nmt_unigram_v1_32000.model | vocab=32000
Targets: [10, 30]
Batch policy: {'10M': {'max_tokens': 9000, 'grad_accum': 2}, '30M': {'max_tokens': 6200, 'grad_accum': 4}, '50M': {'max_tokens': 4400, 'grad_accum': 6}, '60M': {'max_tokens': 3200, 'grad_accum': 8}}
Perf target: 100,000 tok/s | OOM retries/model: 6 | workers=8


In [20]:
def count_parameters(model: nn.Module) -> dict:
    # Counts how large the model is (important for comparing model sizes)
    # Total parameters in the model (all weights)
    total = sum(p.numel() for p in model.parameters())
    
    # Trainable parameters (only those updated during training)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # Return both values for analysis
    return {"total": total, "trainable": trainable}


def budget_report(model_name: str, target_m: int, actual_params: int) -> dict:
    # Compares expected model size vs actual parameter count
    # Convert target from millions to exact number
    target_params = int(target_m * 1_000_000)
    
    # Calculate difference between actual and target
    abs_err = actual_params - target_params
    
    # Percentage error (how far we are from target size)
    pct_err = (abs_err / target_params) * 100.0
    
    # Store results in structured format
    report = {
        "model": model_name,
        "target_m": target_m,
        "target_params": target_params,
        "actual_params": actual_params,
        "abs_error": abs_err,
        "pct_error": pct_err,
    }
    
    # Print summary → useful during training/debugging
    print(
        f"{model_name}: target={target_params:,} | actual={actual_params:,} | "
        f"error={abs_err:+,} ({pct_err:+.2f}%)"
    )
    
    return report

In [21]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 1024, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # Create positional encoding matrix to inject word order information
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()

        # Different frequencies for each dimension
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        # Register as buffer (non-trainable but saved with the model)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Add positional encoding to embeddings
        return self.dropout(x + self.pe[:, : x.size(1)])


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, nhead: int, dropout: float):
        super().__init__()
        if d_model % nhead != 0:
            raise ValueError("d_model must be divisible by nhead")

        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.qkv = nn.Linear(d_model, d_model * 3, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, key_padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        bsz, seq_len, d_model = x.size()
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(bsz, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        k = k.view(bsz, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        v = v.view(bsz, seq_len, self.nhead, self.head_dim).transpose(1, 2)

        attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_scores = attn_scores.masked_fill(mask, float("-inf"))

        attn = torch.softmax(attn_scores, dim=-1)
        attn = self.attn_drop(attn)

        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(bsz, seq_len, d_model)
        out = self.out_proj(out)
        return self.proj_drop(out)


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.act = nn.GELU()
        self.drop1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)
        x = self.fc2(x)
        return self.drop2(x)


class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, nhead: int, d_ff: int, dropout: float):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadSelfAttention(d_model, nhead, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)

    def forward(self, x: torch.Tensor, src_key_padding_mask: torch.Tensor) -> torch.Tensor:
        x = x + self.self_attn(self.norm1(x), key_padding_mask=src_key_padding_mask)
        x = x + self.ffn(self.norm2(x))
        return x


class EncoderStack(nn.Module):
    def __init__(self, d_model: int, nhead: int, d_ff: int, num_layers: int, dropout: float):
        super().__init__()
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, nhead, d_ff, dropout) for _ in range(num_layers)]
        )

    def forward(self, x: torch.Tensor, src_key_padding_mask: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, src_key_padding_mask=src_key_padding_mask)
        return x


class DecoderStack(nn.Module):
    def __init__(self, d_model: int, nhead: int, d_ff: int, num_layers: int, dropout: float):
        super().__init__()

        # One decoder layer: self-attention + cross-attention + feedforward
        layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )

        # Stack multiple decoder layers
        self.decoder = nn.TransformerDecoder(layer, num_layers=num_layers)

    def forward(
        self,
        tgt: torch.Tensor,
        memory: torch.Tensor,
        tgt_mask: torch.Tensor,
        tgt_key_padding_mask: torch.Tensor,
        memory_key_padding_mask: torch.Tensor,
    ) -> torch.Tensor:
        # Generate output using encoder outputs (memory)
        return self.decoder(
            tgt=tgt,
            memory=memory,
            tgt_mask=tgt_mask,  # prevents seeing future tokens
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask,
        )


class NMTTrans(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        nhead: int,
        num_enc_layers: int,
        num_dec_layers: int,
        d_ff: int,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.d_model = d_model
        self.scale = math.sqrt(d_model)  # scale embeddings (stabilizes training)

        # Convert token IDs to dense vectors
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)

        # Add positional information
        self.position = PositionalEncoding(d_model=d_model, max_len=MAX_SEQ_LEN, dropout=dropout)

        # Encoder and decoder
        self.encoder = EncoderStack(
            d_model=d_model,
            nhead=nhead,
            d_ff=d_ff,
            num_layers=num_enc_layers,
            dropout=dropout,
        )
        self.decoder = DecoderStack(
            d_model=d_model,
            nhead=nhead,
            d_ff=d_ff,
            num_layers=num_dec_layers,
            dropout=dropout,
        )

        # Final layer maps to vocabulary logits
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying reduces parameters and improves performance
        self.lm_head.weight = self.embedding.weight

        self._init_weights()

    def _init_weights(self) -> None:
        # Xavier initialization for stable gradients
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
        # Create masks to ignore padding tokens
        src_pad_mask = src.eq(PAD_ID)
        tgt_pad_mask = tgt.eq(PAD_ID)

        # Prevent decoder from accessing future words (causal masking)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1), device=tgt.device)

        # Convert tokens to embeddings and add positional info
        src_emb = self.position(self.embedding(src) * self.scale)
        tgt_emb = self.position(self.embedding(tgt) * self.scale)

        # Encoder processes source sentence
        memory = self.encoder(src_emb, src_key_padding_mask=src_pad_mask)

        # Decoder generates output using encoder memory
        decoded = self.decoder(
            tgt=tgt_emb,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_pad_mask,
            memory_key_padding_mask=src_pad_mask,
        )

        # Output probabilities over vocabulary
        return self.lm_head(decoded)

In [22]:
MODEL_CONFIGS = {
    "10M": dict(d_model=256, nhead=4, num_enc_layers=3, num_dec_layers=3, d_ff=1024, dropout=0.10),
    "30M": dict(d_model=512, nhead=8, num_enc_layers=4, num_dec_layers=4, d_ff=2048, dropout=0.10),
    "50M": dict(d_model=512, nhead=8, num_enc_layers=6, num_dec_layers=6, d_ff=2304, dropout=0.10),
    "60M": dict(d_model=640, nhead=8, num_enc_layers=6, num_dec_layers=6, d_ff=2560, dropout=0.12),
}

model_10m = NMTTrans(vocab_size=VOCAB_SIZE, **MODEL_CONFIGS["10M"]).to(DEVICE)
params_10m = count_parameters(model_10m)["total"]
report_10m = budget_report("10M", 10, params_10m)

10M: target=10,000,000 | actual=13,718,528 | error=+3,718,528 (+37.19%)


In [23]:
model_30m = NMTTrans(vocab_size=VOCAB_SIZE, **MODEL_CONFIGS["30M"]).to(DEVICE)
params_30m = count_parameters(model_30m)["total"]
report_30m = budget_report("30M", 30, params_30m)

30M: target=30,000,000 | actual=45,801,472 | error=+15,801,472 (+52.67%)


In [24]:
model_50m = NMTTrans(vocab_size=VOCAB_SIZE, **MODEL_CONFIGS["50M"]).to(DEVICE)
params_50m = count_parameters(model_50m)["total"]
report_50m = budget_report("50M", 50, params_50m)

50M: target=50,000,000 | actual=63,659,008 | error=+13,659,008 (+27.32%)


In [25]:
model_60m = NMTTrans(vocab_size=VOCAB_SIZE, **MODEL_CONFIGS["60M"]).to(DEVICE)
params_60m = count_parameters(model_60m)["total"]
report_60m = budget_report("60M", 60, params_60m)

60M: target=60,000,000 | actual=89,400,320 | error=+29,400,320 (+49.00%)


In [26]:
def auto_adjust_config(model_name: str, cfg: dict, target_m: int, tolerance_pct: float = 2.0) -> Tuple[Dict, Dict]:
    tuned = dict(cfg)
    for _ in range(12):
        m = NMTTrans(vocab_size=VOCAB_SIZE, **tuned)
        actual = count_parameters(m)["total"]
        del m
        gc.collect()

        rpt = budget_report(model_name, target_m, actual)
        if abs(rpt["pct_error"]) <= tolerance_pct:
            return tuned, rpt

        if actual < rpt["target_params"]:
            tuned["d_ff"] += 128
        else:
            tuned["d_ff"] = max(512, tuned["d_ff"] - 128)

    final_model = NMTTrans(vocab_size=VOCAB_SIZE, **tuned)
    final_report = budget_report(model_name, target_m, count_parameters(final_model)["total"])
    del final_model
    return tuned, final_report


validated_configs = {}
param_reports = []
for m in TARGET_MILLIONS:
    name = f"{m}M"
    tuned_cfg, rpt = auto_adjust_config(name, MODEL_CONFIGS[name], target_m=m, tolerance_pct=2.5)
    validated_configs[name] = tuned_cfg
    param_reports.append(rpt)

pd.DataFrame(param_reports)


10M: target=10,000,000 | actual=13,718,528 | error=+3,718,528 (+37.19%)
10M: target=10,000,000 | actual=13,324,544 | error=+3,324,544 (+33.25%)
10M: target=10,000,000 | actual=12,930,560 | error=+2,930,560 (+29.31%)
10M: target=10,000,000 | actual=12,536,576 | error=+2,536,576 (+25.37%)
10M: target=10,000,000 | actual=12,142,592 | error=+2,142,592 (+21.43%)
10M: target=10,000,000 | actual=12,142,592 | error=+2,142,592 (+21.43%)
10M: target=10,000,000 | actual=12,142,592 | error=+2,142,592 (+21.43%)
10M: target=10,000,000 | actual=12,142,592 | error=+2,142,592 (+21.43%)
10M: target=10,000,000 | actual=12,142,592 | error=+2,142,592 (+21.43%)
10M: target=10,000,000 | actual=12,142,592 | error=+2,142,592 (+21.43%)
10M: target=10,000,000 | actual=12,142,592 | error=+2,142,592 (+21.43%)
10M: target=10,000,000 | actual=12,142,592 | error=+2,142,592 (+21.43%)
10M: target=10,000,000 | actual=12,142,592 | error=+2,142,592 (+21.43%)
30M: target=30,000,000 | actual=45,801,472 | error=+15,801,472 (

,model,target_m,target_params,actual_params,abs_error,pct_error
0,10M,10,10000000,12142592,2142592,21.425920
1,30M,30,30000000,33206272,3206272,10.687573


In [27]:
class NMTDataset(Dataset):
    def __init__(self, src_mmap: np.ndarray, tgt_mmap: np.ndarray, lengths: np.ndarray):
        # Memory-mapped numpy array containing source token IDs. Memory mapping avoids loading
        # the entire array into RAM, instead reading from disk as needed.
        self.src = src_mmap
        # Memory-mapped numpy array containing target token IDs.
        self.tgt = tgt_mmap
        # Array containing the true sequence lengths for each source and target sentence pair.
        self.lengths = lengths

    def __len__(self) -> int:
        # Returns the total number of sentence pairs in the dataset.
        return len(self.lengths)

    def __getitem__(self, idx: int):
        # Get the actual lengths of the source and target sequences for this specific index.
        sl, tl = int(self.lengths[idx, 0]), int(self.lengths[idx, 1])
        # Read the sequence from the memory-mapped array, copy it to memory, and convert to a PyTorch tensor.
        # We only slice up to the true sequence length (sl, tl) to ignore empty padding values.
        src = torch.from_numpy(self.src[idx, :sl].copy()).long()
        tgt = torch.from_numpy(self.tgt[idx, :tl].copy()).long()
        return src, tgt


class BucketBatchSampler(Sampler):
    def __init__(self, lengths: np.ndarray, max_tokens: int, shuffle: bool = True):
        # The maximum total number of tokens allowed in a single batch (batch_size * max_seq_length).
        self.max_tokens = max_tokens
        # Whether to randomly shuffle the order of the *batches* before each epoch.
        self.shuffle = shuffle
        # For each sentence pair, calculate the maximum length between the source and target.
        # This defines the "bounding box" or padding required for that pair.
        self.max_lens = np.maximum(lengths[:, 0], lengths[:, 1])
        # Sort the dataset indices by sequence length. Grouping similar lengths minimizes padding waste.
        self.sorted_indices = np.argsort(self.max_lens)
        # Pre-compute the exact data indices that form each batch using the lengths and token limit.
        self.batches = self._build_batches()

    def _build_batches(self):
        batches = []
        cur_batch, cur_max_len = [], 0
        # Iterate over the dataset indices, ordered from shortest to longest sequences.
        for idx in self.sorted_indices:
            # The length of the current sequence pair.
            seq_len = int(self.max_lens[idx])
            # The maximum sequence length of the batch if we decide to add this sequence.
            candidate_max = max(cur_max_len, seq_len)
            
            # Check if adding this sequence would cause the batch's total padded token count 
            # (number of sentences * max_length_in_batch) to exceed the allowed limit.
            if (len(cur_batch) + 1) * candidate_max > self.max_tokens:
                # The batch is "full". Save it to our list of batches.
                if cur_batch:
                    batches.append(cur_batch)
                # Start a new batch containing only the current sequence.
                cur_batch, cur_max_len = [int(idx)], seq_len
            else:
                # It fits! Add the index to the current batch and update the max length.
                cur_batch.append(int(idx))
                cur_max_len = candidate_max
                
        # After the loop finishes, append any remaining sequences as the final batch.
        if cur_batch:
            batches.append(cur_batch)
        return batches

    def __iter__(self):
        if self.shuffle:
            random.shuffle(self.batches)
        for batch in self.batches:
            yield batch

    def __len__(self) -> int:
        return len(self.batches)


def collate_fn(batch):
    src, tgt = zip(*batch)
    src_max = max(s.size(0) for s in src)
    tgt_max = max(t.size(0) for t in tgt)
    src_pad = torch.full((len(batch), src_max), PAD_ID, dtype=torch.long)
    tgt_pad = torch.full((len(batch), tgt_max), PAD_ID, dtype=torch.long)
    for i, (s, t) in enumerate(zip(src, tgt)):
        src_pad[i, : s.size(0)] = s
        tgt_pad[i, : t.size(0)] = t
    return src_pad, tgt_pad



In [28]:
cache_tag = "unigram_v1_32000"
train_src_path = CACHE_DIR / f"train_{cache_tag}_src.npy"
train_tgt_path = CACHE_DIR / f"train_{cache_tag}_tgt.npy"
train_lens_path = CACHE_DIR / f"train_{cache_tag}_lens.npy"
val_src_path = CACHE_DIR / f"val_{cache_tag}_src.npy"
val_tgt_path = CACHE_DIR / f"val_{cache_tag}_tgt.npy"
val_lens_path = CACHE_DIR / f"val_{cache_tag}_lens.npy"
cache_paths = [train_src_path, train_tgt_path, train_lens_path, val_src_path, val_tgt_path, val_lens_path]

if all(path.exists() for path in cache_paths):
    print("Cache already exists; skipping rebuild.")
else:
    import pyarrow.parquet as pq

    parquet_files = sorted(DATA_DIR.glob("train-*.parquet"))
    if not parquet_files:
        raise FileNotFoundError("No parquet shards found under the data directory.")

    total_rows = sum(pq.ParquetFile(str(path)).metadata.num_rows for path in parquet_files)
    val_size = min(4928, total_rows)
    train_rows = total_rows - val_size

    def encode_example(text):
        # Encode text to token IDs, adding Start-Of-Sequence and End-Of-Sequence tags.
        # Truncate to MAX_SEQ_LEN - 2 to leave room for BOS and EOS.
        token_ids = [BOS_ID] + sp.encode(str(text), out_type=int)[: MAX_SEQ_LEN - 2] + [EOS_ID]
        token_ids = token_ids[:MAX_SEQ_LEN]
        
        # Create a uniform row filled with PAD_ID. This standardizes the shape for on-disk storage.
        row = np.full(MAX_SEQ_LEN, PAD_ID, dtype=np.int32)
        row[:len(token_ids)] = token_ids
        
        # Return the padded row and the actual sequence length (used later to slice off padding).
        return row, len(token_ids)

    # Create memory-mapped files on disk. This allows us to write and read a massive
    # dataset matrix without requiring enough RAM to hold it all at once.
    train_src = np.lib.format.open_memmap(str(train_src_path), mode="w+", dtype=np.int32, shape=(train_rows, MAX_SEQ_LEN))
    train_tgt = np.lib.format.open_memmap(str(train_tgt_path), mode="w+", dtype=np.int32, shape=(train_rows, MAX_SEQ_LEN))
    train_lens = np.lib.format.open_memmap(str(train_lens_path), mode="w+", dtype=np.int32, shape=(train_rows, 2))
    val_src = np.lib.format.open_memmap(str(val_src_path), mode="w+", dtype=np.int32, shape=(val_size, MAX_SEQ_LEN))
    val_tgt = np.lib.format.open_memmap(str(val_tgt_path), mode="w+", dtype=np.int32, shape=(val_size, MAX_SEQ_LEN))
    val_lens = np.lib.format.open_memmap(str(val_lens_path), mode="w+", dtype=np.int32, shape=(val_size, 2))

    global_idx = 0
    train_idx = 0
    val_idx = 0

    for parquet_path in parquet_files:
        parquet_file = pq.ParquetFile(str(parquet_path))
        # Read the parquet files in chunks (batches) to remain memory efficient.
        for batch in parquet_file.iter_batches(batch_size=8192, columns=["src", "tgt"]):
            src_values = batch.column(0).to_pylist()
            tgt_values = batch.column(1).to_pylist()
            
            # Process each text pair, pad it for standard storage, and insert it into the memory-mapped array.
            for src_text, tgt_text in zip(src_values, tgt_values):
                src_row, src_len = encode_example(src_text)
                tgt_row, tgt_len = encode_example(tgt_text)
                if global_idx < train_rows:
                    train_src[train_idx] = src_row
                    train_tgt[train_idx] = tgt_row
                    train_lens[train_idx] = (src_len, tgt_len)
                    train_idx += 1
                else:
                    val_src[val_idx] = src_row
                    val_tgt[val_idx] = tgt_row
                    val_lens[val_idx] = (src_len, tgt_len)
                    val_idx += 1
                global_idx += 1

    train_src.flush()
    train_tgt.flush()
    train_lens.flush()
    val_src.flush()
    val_tgt.flush()
    val_lens.flush()

    print(f"Created cache: train={train_idx:,} | val={val_idx:,} | total={global_idx:,}")

Cache already exists; skipping rebuild.


In [29]:
# All parquet files are treated as training corpus.
parquet_files = sorted(DATA_DIR.glob("train-*.parquet"))
print(f"Detected parquet training files: {len(parquet_files)}")
for p in parquet_files:
    print(f"  - {p.name}")

# The val cache below is a held-out split derived from that same training corpus.
cache_tag = "unigram_v1_32000"
train_src = np.load(str(CACHE_DIR / f"train_{cache_tag}_src.npy"), mmap_mode="r")
train_tgt = np.load(str(CACHE_DIR / f"train_{cache_tag}_tgt.npy"), mmap_mode="r")
train_lens = np.load(str(CACHE_DIR / f"train_{cache_tag}_lens.npy"))
val_src = np.load(str(CACHE_DIR / f"val_{cache_tag}_src.npy"), mmap_mode="r")
val_tgt = np.load(str(CACHE_DIR / f"val_{cache_tag}_tgt.npy"), mmap_mode="r")
val_lens = np.load(str(CACHE_DIR / f"val_{cache_tag}_lens.npy"))

train_dataset = NMTDataset(train_src, train_tgt, train_lens)
val_dataset = NMTDataset(val_src, val_tgt, val_lens)

print(f"Loaded cached arrays: train={len(train_dataset):,} | val={len(val_dataset):,}")

Detected parquet training files: 3
  - train-00000-of-00003.parquet
  - train-00001-of-00003.parquet
  - train-00002-of-00003.parquet
Loaded cached arrays: train=4,941,107 | val=4,928


In [30]:
# --- Data Loading Utilities ---
def make_loaders(max_tokens: int):
    # Configures PyTorch DataLoaders using our sequence-length bucket sampler and dynamic padding collation
    workers = int(TRAINING_DEFAULTS["num_workers"])
    workers = max(0, min(workers, os.cpu_count() or 1))
    pin = DEVICE.type == "cuda"
    common_kwargs = {
        "collate_fn": collate_fn,
        "num_workers": workers,
        "pin_memory": pin,
        "persistent_workers": workers > 0,
    }
    if workers > 0:
        common_kwargs["prefetch_factor"] = int(TRAINING_DEFAULTS["prefetch_factor"])

    train_loader = DataLoader(
        train_dataset,
        batch_sampler=BucketBatchSampler(train_lens, max_tokens=max_tokens, shuffle=True),
        **common_kwargs,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_sampler=BucketBatchSampler(val_lens, max_tokens=max_tokens, shuffle=False),
        **common_kwargs,
    )
    return train_loader, val_loader


# --- Memory Management & OOM Recovery Utilities ---
def clear_runtime_memory():
    # Force Python garbage collection and empty PyTorch's pre-allocated CUDA cache, helpful post-OOM
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


def is_oom_error(exc: RuntimeError) -> bool:
    # Helper to aggressively detect Out-of-Memory exceptions, distinguishing them from logic errors
    msg = str(exc).lower()
    if (
        "out of memory" in msg
        or "cuda error: out of memory" in msg
        or "cudnn_status_alloc_failed" in msg
        or "oom in" in msg
        or "oom during" in msg
        or "cublas_status" in msg
        or "cublas" in msg
    ):
        return True
    if exc.__cause__ is not None and isinstance(exc.__cause__, RuntimeError):
        cause_msg = str(exc.__cause__).lower()
        return (
            "out of memory" in cause_msg
            or "cuda error: out of memory" in cause_msg
            or "cudnn_status_alloc_failed" in cause_msg
            or "cublas_status" in cause_msg
            or "cublas" in cause_msg
        )
    return False


# --- Transformer Training Schedule Utilities ---
def make_scheduler(optimizer, warmup_steps: int):
    # Implements the standard Transformer learning rate schedule: linear warmup followed by inverse square-root decay
    def lr_lambda(step: int):
        step = max(step, 1)
        return min(step ** (-0.5), step * warmup_steps ** (-1.5)) * (warmup_steps ** 0.5)

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def safe_ppl(loss_value: float) -> float:
    # Calculates perplexity safely, clamping the math limit to avoid 'inf' overflow on very high early loss
    return math.exp(min(loss_value, 20.0))


# Setup Mixed Precision (AMP) variables. bfloat16 is preferred on Ampere+ GPUs for stability, falling back to float16
AMP_DTYPE = torch.bfloat16 if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16


def amp_ctx():
    # Returns the precision context manager based on device capabilities via modern torch.amp
    enabled = TRAINING_DEFAULTS["use_amp"] and DEVICE.type == "cuda"
    if not enabled:
        return nullcontext()
    return torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE)


def make_grad_scaler():
    # Creates the Gradient Scaler needed to prevent underflow specifically when using float16 mixed precision.
    enabled = TRAINING_DEFAULTS["use_amp"] and DEVICE.type == "cuda" and AMP_DTYPE == torch.float16
    return torch.amp.GradScaler("cuda", enabled=enabled)


def choose_grad_accum(base_max_tokens: int, base_grad_accum: int, current_max_tokens: int) -> int:
    # Called if the VRAM limits are exceeded. When max_tokens shrinks, this increases 
    # accumulation steps to maintain the target "global effective batch size".
    target_effective_tokens = max(1, base_max_tokens * base_grad_accum)
    adjusted = math.ceil(target_effective_tokens / max(current_max_tokens, 1))
    return max(1, min(TRAINING_DEFAULTS["max_grad_accum"], adjusted))


def build_optimizer(model):
    # Initializes standard AdamW, prioritizing the heavily optimized CUDA 'fused' execution path
    kwargs = {
        "lr": TRAINING_DEFAULTS["lr"],
        "betas": (0.9, 0.98),
        "eps": 1e-9,
        "weight_decay": TRAINING_DEFAULTS["weight_decay"],
    }
    if DEVICE.type == "cuda":
        return torch.optim.AdamW(model.parameters(), fused=True, **kwargs), "fused"
    return torch.optim.AdamW(model.parameters(), **kwargs), "standard"

In [31]:
def train_model(
    name: str,
    model,
    train_loader,
    val_loader,
    optimizer,
    train_criterion,
    eval_criterion,
    scaler,
    scheduler,
    grad_accum: int,
    epochs: int,
):
    # Train for multiple epochs with AMP, gradient accumulation, and throughput logging.
    best_val = float("inf")
    best_epoch = 0
    start = time.time()
    total_tokens_all_epochs = 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_tokens = 0
        processed_samples = 0

        optimizer.zero_grad(set_to_none=True)
        t0 = time.time()
        interval_t0 = t0
        interval_tokens = 0

        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()

        max_train_batches = min(TRAINING_DEFAULTS["max_train_batches_per_epoch"], len(train_loader))
        log_interval = max(1, max_train_batches // 10)

        for i, (src, tgt) in enumerate(train_loader, start=1):
            if i > max_train_batches:
                break

            if tgt.size(1) < 2:
                continue  # Skip empty/too-short targets.

            try:
                src = src.to(DEVICE, non_blocking=True)
                tgt = tgt.to(DEVICE, non_blocking=True)
                tgt_input, tgt_label = tgt[:, :-1], tgt[:, 1:]

                with amp_ctx():
                    logits = model(src, tgt_input)
                    # Normalize loss for gradient accumulation.
                    loss = train_criterion(logits.reshape(-1, logits.size(-1)), tgt_label.reshape(-1)) / grad_accum

                if not torch.isfinite(loss.detach()):
                    raise RuntimeError(f"Non-finite loss in {name} epoch={epoch} batch={i}")

            except RuntimeError as exc:
                if is_oom_error(exc):
                    clear_runtime_memory()
                    raise RuntimeError(f"OOM in {name} epoch={epoch} batch={i}") from exc
                raise

            try:
                scaler.scale(loss).backward()
            except RuntimeError as exc:
                if is_oom_error(exc):
                    clear_runtime_memory()
                    raise RuntimeError(f"OOM during backward in {name} epoch={epoch} batch={i}") from exc
                raise

            if i % grad_accum == 0 or i == max_train_batches:
                # Unscale, clip, and step optimizer at accumulation boundary.
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), TRAINING_DEFAULTS["grad_clip"])
                try:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    scheduler.step()
                except RuntimeError as exc:
                    if is_oom_error(exc):
                        clear_runtime_memory()
                        raise RuntimeError(f"OOM during optimizer step in {name} epoch={epoch} batch={i}") from exc
                    raise

            n_tokens = int((tgt_label != PAD_ID).sum().item())
            if n_tokens <= 0:
                continue

            batch_samples = src.size(0)
            total_loss += loss.item() * grad_accum * n_tokens
            total_tokens += n_tokens
            interval_tokens += n_tokens
            processed_samples += batch_samples

            if i % log_interval == 0 or i == 1 or i == max_train_batches:
                avg_loss = total_loss / max(total_tokens, 1)
                current_lr = optimizer.param_groups[0]["lr"]

                now = time.time()
                elapsed_interval = max(1e-3, now - interval_t0)
                tokens_per_sec = interval_tokens / elapsed_interval
                speed_flag = "OK" if tokens_per_sec >= TRAINING_DEFAULTS["target_tokens_per_sec"] else "LOW"

                print(
                    f"    batch {i:,}/{max_train_batches:,} | batch_size={batch_samples} | samples={processed_samples:,} | "
                    f"loss={avg_loss:.4f} | ppl={safe_ppl(avg_loss):.2f} | "
                    f"lr={current_lr:.2e} | throughput={tokens_per_sec:,.0f} tok/s ({speed_flag})"
                )

                interval_t0 = time.time()
                interval_tokens = 0

        avg_loss = total_loss / max(total_tokens, 1)
        tr_time = time.time() - t0
        epoch_tokens_per_sec = total_tokens / max(tr_time, 1e-6)

        # Validation pass to track best epoch.
        try:
            val_loss = evaluate(model, val_loader, eval_criterion)
        except RuntimeError as exc:
            if is_oom_error(exc):
                clear_runtime_memory()
                raise RuntimeError(f"OOM during validation in {name} epoch={epoch}") from exc
            raise

        train_ppl = safe_ppl(avg_loss)
        val_ppl = safe_ppl(val_loss)
        peak_mem_gb = (torch.cuda.max_memory_allocated() / 1e9) if DEVICE.type == "cuda" else 0.0

        speed_flag = "OK" if epoch_tokens_per_sec >= TRAINING_DEFAULTS["target_tokens_per_sec"] else "LOW"
        print(
            f"{name} | epoch={epoch} | train_loss={avg_loss:.4f} | train_ppl={train_ppl:.2f} | "
            f"val_loss={val_loss:.4f} | val_ppl={val_ppl:.2f} | "
            f"epoch_tps={epoch_tokens_per_sec:,.0f} ({speed_flag}) | peak_mem={peak_mem_gb:.2f} GB | epoch_time={tr_time:.1f}s"
        )

        total_tokens_all_epochs += total_tokens

        if val_loss < best_val:
            best_val = val_loss
            best_epoch = epoch

    elapsed = time.time() - start
    avg_tokens_per_sec = total_tokens_all_epochs / max(elapsed, 1e-6)
    return best_epoch, best_val, elapsed, avg_tokens_per_sec


@torch.inference_mode()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    max_val_batches = min(TRAINING_DEFAULTS["max_val_batches"], len(loader))

    for i, (src, tgt) in enumerate(loader, start=1):
        if i > max_val_batches:
            break

        if tgt.size(1) < 2:
            continue

        src = src.to(DEVICE, non_blocking=True)
        tgt = tgt.to(DEVICE, non_blocking=True)
        tgt_input, tgt_label = tgt[:, :-1], tgt[:, 1:]

        with amp_ctx():
            logits = model(src, tgt_input)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_label.reshape(-1))

        n_tokens = int((tgt_label != PAD_ID).sum().item())
        if n_tokens <= 0:
            continue

        total_loss += loss.item() * n_tokens
        total_tokens += n_tokens

    return total_loss / max(total_tokens, 1)


@torch.inference_mode()
def greedy_decode(model, src_text, sp_model, max_len=100):
    model.eval()
    m = model._orig_mod if hasattr(model, "_orig_mod") else model

    # Tokenization happens here: raw text is converted to SentencePiece token IDs.
    src_ids = [BOS_ID] + sp_model.encode(src_text)[: MAX_SEQ_LEN - 2] + [EOS_ID]
    src_tensor = torch.tensor([src_ids], dtype=torch.long, device=DEVICE)
    src_pad_mask = src_tensor == PAD_ID

    src_emb = m.position(m.embedding(src_tensor) * m.scale)
    memory = m.encoder(src_emb, src_key_padding_mask=src_pad_mask)

    tgt_ids = [BOS_ID]
    for _ in range(max_len):
        tgt_tensor = torch.tensor([tgt_ids], dtype=torch.long, device=DEVICE)
        tgt_pad_mask = tgt_tensor == PAD_ID
        tgt_emb = m.position(m.embedding(tgt_tensor) * m.scale)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(len(tgt_ids), device=DEVICE)
        out = m.decoder(
            tgt=tgt_emb,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_pad_mask,
            memory_key_padding_mask=src_pad_mask,
        )
        logits = m.lm_head(out[:, -1, :])
        next_id = logits.argmax(dim=-1).item()
        if next_id == EOS_ID:
            break
        tgt_ids.append(next_id)

    return sp_model.decode(tgt_ids[1:])


def run_sample_inferences(trained_model, sample_texts):
    print("\n" + "=" * 60)
    print("Sample Inferences")
    print("=" * 60)
    for src_text in sample_texts:
        translation = greedy_decode(trained_model, src_text, sp)
        print(f"\nEN: {src_text}")
        print(f"TE: {translation}")

In [32]:
def train_and_save_all_models():
    all_results = []  # stores final results of all models

    # Print dataset info (helps understand training scale)
    train_samples = len(train_dataset)
    val_samples = len(val_dataset)
    print("\n=== Dataset Summary Before Training ===")
    print(f"Training samples: {train_samples:,}")
    print(f"Validation samples: {val_samples:,}")
    print(f"Total samples: {train_samples + val_samples:,}")
    print(f"AMP dtype: {AMP_DTYPE}")  # precision used (important for speed)

    # Loop over different model sizes (10M, 30M, 50M, 60M)
    for target_m in TARGET_MILLIONS:
        name = f"{target_m}M"
        cfg = validated_configs[name]  # tuned config matching parameter budget

        # Batch policy → controls memory usage
        base_policy = dict(BATCH_POLICY[name])
        base_max_tokens = int(base_policy["max_tokens"])
        base_grad_accum = int(base_policy["grad_accum"])

        # Initial training settings
        max_tokens = base_max_tokens
        grad_accum = base_grad_accum
        attempts = 0  # number of retries if OOM occurs

        print("\n" + "=" * 72)
        print(f"Training {name} | cfg={cfg} | initial_policy={base_policy}")
        print("=" * 72)

        # Retry loop → handles GPU memory errors dynamically
        while True:
            # Initialize variables (helps safe cleanup on failure)
            model = None
            optimizer = None
            train_criterion = None
            eval_criterion = None
            scaler = None
            scheduler = None
            train_loader = None
            val_loader = None

            try:
                print(f"Attempt {attempts + 1}: max_tokens={max_tokens}, grad_accum={grad_accum}")

                # Create dataloaders (handles batching and padding)
                train_loader, val_loader = make_loaders(max_tokens=max_tokens)

                print(f"Samples: train={len(train_dataset):,} | val={len(val_dataset):,}")
                print(f"Batches: train={len(train_loader):,} | val={len(val_loader):,}")

                # Build model and move to device (GPU/CPU)
                model = NMTTrans(vocab_size=VOCAB_SIZE, **cfg).to(DEVICE)

                # Check parameter count → ensures model matches target size
                counts = count_parameters(model)
                budget = budget_report(name, target_m, counts["total"])

                # Optimizer (AdamW)
                optimizer, optimizer_mode = build_optimizer(model)

                # Loss functions
                train_criterion = nn.CrossEntropyLoss(
                    ignore_index=PAD_ID,
                    label_smoothing=TRAINING_DEFAULTS["label_smoothing"],  # smoother training
                )
                eval_criterion = nn.CrossEntropyLoss(
                    ignore_index=PAD_ID,
                    label_smoothing=0.0,  # exact evaluation
                )

                # Mixed precision + scheduler
                scaler = make_grad_scaler()
                scheduler = make_scheduler(optimizer, warmup_steps=TRAINING_DEFAULTS["warmup_steps"])

                # Main training loop
                best_epoch, best_val, _, avg_tps = train_model(
                    name=name,
                    model=model,
                    train_loader=train_loader,
                    val_loader=val_loader,
                    optimizer=optimizer,
                    train_criterion=train_criterion,
                    eval_criterion=eval_criterion,
                    scaler=scaler,
                    scheduler=scheduler,
                    grad_accum=grad_accum,
                    epochs=TRAINING_DEFAULTS["epochs_per_model"],
                )

                # Store actual batch settings used
                effective_policy = {"max_tokens": max_tokens, "grad_accum": grad_accum}

                break  # training successful → exit retry loop

            except RuntimeError as exc:
                # Handle GPU Out-Of-Memory errors
                if not is_oom_error(exc):
                    raise

                attempts += 1
                print(f"OOM detected for {name}: {exc}")

                # Stop if too many retries
                if attempts > TRAINING_DEFAULTS["oom_retry_limit"]:
                    raise RuntimeError(
                        f"Exceeded OOM retry limit for {name}. "
                        f"Last max_tokens={max_tokens}."
                    ) from exc

                # Reduce batch size to fit memory
                next_max_tokens = max(
                    TRAINING_DEFAULTS["min_max_tokens"],
                    int(max_tokens * TRAINING_DEFAULTS["auto_batch_shrink_factor"]),
                )

                # Ensure reduction actually happens
                if next_max_tokens >= max_tokens:
                    next_max_tokens = max(TRAINING_DEFAULTS["min_max_tokens"], max_tokens - 50)

                if next_max_tokens == max_tokens:
                    raise RuntimeError(
                        f"OOM persists for {name}; cannot shrink max_tokens below {max_tokens}."
                    ) from exc

                max_tokens = next_max_tokens

                # Adjust gradient accumulation → keeps effective batch size stable
                grad_accum = choose_grad_accum(base_max_tokens, base_grad_accum, max_tokens)

                print(
                    f"Retrying {name} with reduced max_tokens={max_tokens}, "
                    f"adjusted grad_accum={grad_accum}"
                )

                # Clean memory before retry
                if model is not None:
                    del model
                if optimizer is not None:
                    del optimizer
                if train_criterion is not None:
                    del train_criterion
                if eval_criterion is not None:
                    del eval_criterion
                if scaler is not None:
                    del scaler
                if scheduler is not None:
                    del scheduler
                if train_loader is not None:
                    del train_loader
                if val_loader is not None:
                    del val_loader

                clear_runtime_memory()

        # Check if training speed meets target
        throughput_ok = avg_tps >= TRAINING_DEFAULTS["target_tokens_per_sec"]

        # Save model checkpoint + metadata
        save_path = CHECKPOINT_DIR / f"final_{target_m}m_modelparametersscratch.pt"
        torch.save(
            {
                "model_name": name,
                "target_m": target_m,
                "config": cfg,
                "param_counts": counts,
                "budget_report": budget,
                "batch_policy": effective_policy,
                "training_defaults": TRAINING_DEFAULTS,
                "optimizer_mode": optimizer_mode,
                "amp_dtype": str(AMP_DTYPE),
                "best_epoch": best_epoch,
                "best_val_loss": best_val,
                "avg_tokens_per_sec": avg_tps,
                "target_met": throughput_ok,
                "oom_retries": attempts,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
            },
            save_path,
        )

        print(f"Saved: {save_path}")

        # Quick sample inference → sanity check for translation quality
        try:
            run_sample_inferences(model, sample_texts)
        except RuntimeError as exc:
            if is_oom_error(exc):
                clear_runtime_memory()
            else:
                raise

        # Store results for comparison
        all_results.append(
            {
                "model": name,
                "best_val_loss": best_val,
                "avg_tokens_per_sec": avg_tps,
            }
        )

        # Free memory before next model
        del model, optimizer, train_loader, val_loader
        clear_runtime_memory()

    # Convert results to DataFrame
    result_df = pd.DataFrame(all_results)

    return result_df

# Define sample_texts before running
sample_texts = [
    "hi", "hello", "thanks", "ok", "yes", "no", "bye", "good morning",
    "They didn’t finish the work", "He saw her duck",
    "I found the book interesting", "They are flying planes",
    "She left him yesterday", "Visiting relatives can be boring",
    "Although the system seems efficient, it fails under heavy load conditions.",
    "If you had told me earlier, I would have helped you.",
    "Despite multiple warnings, he continued to ignore the issue.",
    "The project, which was delayed multiple times, was finally completed.",
    "Even if it rains, we will continue with the event.",

]

# Run full training pipeline
results_df = train_and_save_all_models()
results_df


=== Dataset Summary Before Training ===
Training samples: 4,941,107
Validation samples: 4,928
Total samples: 4,946,035
AMP dtype: torch.bfloat16

Training 10M | cfg={'d_model': 256, 'nhead': 4, 'num_enc_layers': 3, 'num_dec_layers': 3, 'd_ff': 512, 'dropout': 0.1} | initial_policy={'max_tokens': 9000, 'grad_accum': 2}
Attempt 1: max_tokens=9000, grad_accum=2
Samples: train=4,941,107 | val=4,928
Batches: train=8,687 | val=11
10M: target=10,000,000 | actual=12,142,592 | error=+2,142,592 (+21.43%)
    batch 1/7,000 | batch_size=818 | samples=818 | loss=10.4268 | ppl=33753.29 | lr=1.67e-07 | throughput=37,487 tok/s (LOW)
OOM detected for 10M: OOM during backward in 10M epoch=1 batch=494
Retrying 10M with reduced max_tokens=7650, adjusted grad_accum=3
Attempt 2: max_tokens=7650, grad_accum=3
Samples: train=4,941,107 | val=4,928
Batches: train=10,222 | val=12
10M: target=10,000,000 | actual=12,142,592 | error=+2,142,592 (+21.43%)
    batch 1/7,000 | batch_size=1092 | samples=1,092 | loss=10

,model,best_val_loss,avg_tokens_per_sec
0,10M,2.692546,131486.260103
1,30M,2.344930,69912.356586


In [33]:
sample_texts = [
    # 1. Very short / conversational
    "hi",
    "hello",
    "thanks",
    "ok",
    "yes",
    "no",
    "bye",
    "good morning",
    "what's up",
    "see you",

    # 2. Slang / informal
    "bro what are you doing",
    "this is crazy",
    "no way",
    "that’s awesome",
    "I’m kinda tired",
    "gonna go now",
    "lemme check",
    "this sucks",
    "chill man",
    "you serious?",

    # 3. Negation traps
    "I don’t think this will work",
    "He is not unhappy",
    "This is not bad at all",
    "I never said that",
    "They didn’t finish the work",

    # 4. Ambiguous meaning
    "He saw her duck",
    "I found the book interesting",
    "They are flying planes",
    "She left him yesterday",
    "Visiting relatives can be boring",

    # 5. Long + complex
    "Although the system seems efficient, it fails under heavy load conditions.",
    "If you had told me earlier, I would have helped you.",
    "Despite multiple warnings, he continued to ignore the issue.",
    "The project, which was delayed multiple times, was finally completed.",
    "Even if it rains, we will continue with the event.",

    # 6. Questions
    "What are you doing right now?",
    "Why did you say that?",
    "How does this system work?",
    "Can you explain this again?",
    "Where did you learn this?",

    # 7. Numbers / data
    "I have 2 apples and 3 oranges.",
    "The temperature is 35 degrees today.",
    "This costs 500 rupees.",
    "He scored 95 out of 100.",
    "The meeting is at 10:30 AM.",

    # 8. Mixed / tricky grammar
    "Not only did he win, but he also broke the record.",
    "The more you practice, the better you get.",
    "She is smarter than she looks.",
    "Hardly had he arrived when it started raining.",
    "No sooner did we leave than it began to snow.",
]
#Prints a header and loops over TARGET_MILLIONS (10M, 30M, 50M, 60M).
print("Running sample inferences on all saved models...")
for target_m in TARGET_MILLIONS:
    ckpt_path = CHECKPOINT_DIR / f"final_{target_m}m_modelparametersscratch.pt" #Constructs checkpoint path
    if not ckpt_path.exists():
        print(f"Checkpoint for {target_m}M model not found at {ckpt_path}")
        continue

    print(f"\n============================================================")
    print(f"Loading {target_m}M model from {ckpt_path.name}...")
    print(f"============================================================")

    try:
        ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
    except TypeError:
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
    #Builds NMTTrans with saved config, loads its state dict, and sets eval()
    model = NMTTrans(vocab_size=VOCAB_SIZE, **ckpt["config"]).to(DEVICE)#
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    run_sample_inferences(model, sample_texts)

    del model, ckpt
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

Running sample inferences on all saved models...

Loading 10M model from final_10m_modelparametersscratch.pt...

Sample Inferences

EN: hi
TE: ీర

EN: hello
TE: నరకం

EN: thanks
TE: కృతజ్ఞతాస్తుతులు

EN: ok
TE: సరే.

EN: yes
TE: అవును.

EN: no
TE: లేదు.

EN: bye
TE: బై బై

EN: good morning
TE: మంచి ఉదయాన్నే

EN: what's up
TE: ఏంటట?

EN: see you
TE: ీరు చూడు.

EN: bro what are you doing
TE: ఏం చేస్తున్నారో?

EN: this is crazy
TE: ఇది పిచ్చి

EN: no way
TE: మార్గం లేదు

EN: that’s awesome
TE: ఇది అద్భుతం

EN: I’m kinda tired
TE: నేను అలసటతో ఉన్నాను

EN: gonna go now
TE: ఇప్పుడు గొన్న

EN: lemme check
TE: లెమ్మ్ చెక్

EN: this sucks
TE: ఈ గూళ్ళు

EN: chill man
TE: చల్లని మనిషి

EN: you serious?
TE: సీరియస్ గా ఉందా?

EN: I don’t think this will work
TE: ఈ పని చేస్తుందని నేను అనుకోవడం లేదు

EN: He is not unhappy
TE: ఆయన అసంతృప్తితో లేరు

EN: This is not bad at all
TE: ఇది అస్సలు చెడ్డది కాదు

EN: I never said that
TE: నేను ఎప్పుడూ చెప్పలేదు

EN: They didn’t finish the work
TE: వారు పని పూర్

In [36]:
import warnings
from pathlib import Path
import sacrebleu
import torch

# Silence repeated Transformer mask warnings during large decode loops.
warnings.filterwarnings("ignore", category=UserWarning, message=".*key_padding_mask.*")

#helper to reconstruct a saved model 
def load_model(path, config):
    model = NMTTrans(vocab_size=VOCAB_SIZE, **config).to(DEVICE)
    try:
        checkpoint = torch.load(path, map_location=DEVICE, weights_only=True)
    except TypeError:
        checkpoint = torch.load(path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model

#to filter out special tokens and decode the remaining IDs
def decode_clean(token_ids, sp_model):
    clean_ids = [int(t) for t in token_ids if int(t) not in (PAD_ID, BOS_ID, EOS_ID)]
    if not clean_ids:
        return ""
    return sp_model.decode(clean_ids)


@torch.inference_mode()
def compute_bleu(model, val_loader, sp_model, max_sentences=300):
    preds = []
    refs = []

    count = 0
    for src, tgt in val_loader:
        for i in range(src.size(0)):
            src_text = decode_clean(src[i].tolist(), sp_model)
            pred = greedy_decode(model, src_text, sp_model)
            ref = decode_clean(tgt[i].tolist(), sp_model)
            #for each source, decodes the input text, translates with greedy_decode, and decodes the reference
            preds.append(pred)
            refs.append(ref)

            count += 1
            if count >= max_sentences:
                break
        if count >= max_sentences:
            break

    bleu = sacrebleu.corpus_bleu(preds, [refs])
    return bleu.score


CHECKPOINT_DIR = Path("/home/user35/nmt_bundle/cache/checkpoints")

_, val_loader = make_loaders(max_tokens=2048)

models = {
    "10M": (CHECKPOINT_DIR / "final_10m_modelparametersscratch.pt", validated_configs["10M"]),
    "30M": (CHECKPOINT_DIR / "final_30m_modelparametersscratch.pt", validated_configs["30M"]),
}

bleu_scores = {}

for name, (path, cfg) in models.items():
    if not path.exists():
        print(f"{name} checkpoint not found: {path}")
        continue

    model = load_model(path, cfg)
    bleu = compute_bleu(model, val_loader, sp)
    bleu_scores[name] = bleu
    print(f"{name} BLEU: {bleu:.2f}")

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

10M BLEU: 13.16
30M BLEU: 11.16


In [38]:
import warnings
from pathlib import Path

import pandas as pd
import sacrebleu
import torch

# Silence noisy warnings during repeated decoding/model loading.
warnings.filterwarnings("ignore", category=UserWarning, message=".*key_padding_mask.*")
warnings.filterwarnings("ignore", category=UserWarning, message=".*enable_nested_tensor.*")

##helper to reconstruct a saved model
def load_model(path, config):
    model = NMTTrans(vocab_size=VOCAB_SIZE, **config).to(DEVICE)
    try:
        checkpoint = torch.load(path, map_location=DEVICE, weights_only=True)
    except TypeError:
        checkpoint = torch.load(path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model

#to filter out special tokens and decode the remaining IDs
def decode_clean(token_ids, sp_model):
    clean_ids = [int(t) for t in token_ids if int(t) not in (PAD_ID, BOS_ID, EOS_ID)]
    if not clean_ids:
        return ""
    return sp_model.decode(clean_ids)


@torch.inference_mode()
def collect_preds_refs(model, val_loader, sp_model, max_sentences=1500):
    preds = []
    refs = []

    count = 0
    for src, tgt in val_loader:
        for i in range(src.size(0)):
            src_text = decode_clean(src[i].tolist(), sp_model)
            pred = greedy_decode(model, src_text, sp_model)
            ref = decode_clean(tgt[i].tolist(), sp_model)
            #for each source, decodes the input text, translates with greedy_decode, and decodes the reference
            preds.append(pred)
            refs.append(ref)

            count += 1
            if count >= max_sentences:
                return preds, refs
    return preds, refs


def normalize_column(values: pd.Series) -> pd.Series:
    vmin = values.min()
    vmax = values.max()
    if vmax <= vmin:
        return pd.Series([0.0] * len(values), index=values.index)
    return (values - vmin) / (vmax - vmin)


CHECKPOINT_DIR = Path("/home/user35/nmt_bundle/cache/checkpoints")
_, val_loader = make_loaders(max_tokens=2048)

models = {
    "10M": (CHECKPOINT_DIR / "final_10m_modelparametersscratch.pt", validated_configs["10M"]),
    "30M": (CHECKPOINT_DIR / "final_30m_modelparametersscratch.pt", validated_configs["30M"]),
}

EVAL_SENTENCES = 1500
rows = []
#For each model:
#loads the checkpointed model

for name, (path, cfg) in models.items():
    if not path.exists():
        print(f"{name} checkpoint not found: {path}")
        continue
    #character-level matching can reflect translation quality
    model = load_model(path, cfg)
    preds, refs = collect_preds_refs(model, val_loader, sp, max_sentences=EVAL_SENTENCES)
    #computes BLEU and chrF scores
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score

    rows.append(
        {
            "model": name,
            "num_sentences": len(preds),
            "bleu": float(bleu),
            "chrf": float(chrf),
        }
    )

    print(f"{name} | BLEU={bleu:.2f} | chrF={chrf:.2f} | sentences={len(preds)}")

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

score_df = pd.DataFrame(rows)
score_df["bleu_norm"] = normalize_column(score_df["bleu"])
score_df["chrf_norm"] = normalize_column(score_df["chrf"])
score_df["composite"] = 0.5 * score_df["bleu_norm"] + 0.5 * score_df["chrf_norm"]
score_df = score_df.sort_values(by="composite", ascending=False).reset_index(drop=True)

best_model = score_df.loc[0, "model"]
print(f"\nBest model by unbiased composite (BLEU+chrF): {best_model}")

score_df

10M | BLEU=11.99 | chrF=41.73 | sentences=1500
30M | BLEU=12.90 | chrF=42.74 | sentences=1500

Best model by unbiased composite (BLEU+chrF): 30M


,model,num_sentences,bleu,chrf,bleu_norm,chrf_norm,composite
0,30M,1500,12.904829,42.737453,1.0,1.0,1.0
1,10M,1500,11.992708,41.732802,0.0,0.0,0.0
